In [ ]:
import pandas as pd
import re

In [ ]:
df = pd.read_csv('data/cleaned/general_stores_sorted_from_excelsheets.csv')

In [ ]:
# make date columns datetime format so I can sort them

date_cols = [
    "DateOfIncorporation",
    "DissolvedDate",
    "TerminationDate",
    "ExpirationDate",
]

df[date_cols] = df[date_cols].apply(pd.to_datetime)

### Q1: When were the most stores dissolved or terminated?

In [ ]:
# make a sub-df with just those stores that have dissolved or terminated statuses
dissolved_or_terminated_rows = df[
    (df["BusinessStatus"] == "Dissolved")
    | (df["BusinessStatus"] == "Terminated")
]

Now combine all of the dates into a series, drop null values, and group them

In [ ]:
# now make series of all disolved or terminated dates
# by stacking the two datetime series vertically into one sequence
dissolved_or_terminated_dates = pd.concat([
    dissolved_or_terminated_rows["DissolvedDate"],
    dissolved_or_terminated_rows["TerminationDate"],
])

# clean the combined series by dropping nulls
clean_dissolved_or_terminated_dates = dissolved_or_terminated_dates.dropna()

# create decade groups
dissolved_or_terminated_decade_start = (clean_dissolved_or_terminated_dates.dt.year // 10) * 10
dissolved_or_terminated_halfdecade_start = (clean_dissolved_or_terminated_dates.dt.year // 5) * 5

# create decade groups
# // divides two numbers and rounds the result down to the nearest whole integer, so for ex. 1972 --> 1970. That means the "1970" bucket will represent 1970-1980
dissolved_or_terminated_decade_range = dissolved_or_terminated_decade_start.astype(str) + '-' + ((dissolved_or_terminated_decade_start + 9).astype(str))
dissolved_or_terminated_halfdecade_range = dissolved_or_terminated_halfdecade_start.astype(str) + '-' + ((dissolved_or_terminated_halfdecade_start + 4).astype(str))




In [ ]:
dissolved_or_terminated_decade_range.value_counts()
dissolved_or_terminated_halfdecade_range.value_counts()

Now export just that information about frequency of year terminated/dissolved

In [ ]:
# Count by year
year_counts = (
    dissolved_or_terminated_dates.dt.year
    .value_counts()
    .sort_index()
    .rename_axis("years")
    .reset_index(name="count")
)

# Export to CSV
year_counts.to_csv("docs/dissolved_terminated_by_year.csv", index=False)


Now exporting same info but in decade and half decade form for use as needed in story

In [ ]:
decade_counts = (
    dissolved_or_terminated_decade_range
    .value_counts()
    .sort_index()
    .rename_axis("years")
    .reset_index(name="count")
)

decade_counts.to_csv("docs/dissolved_terminated_by_decade.csv", index=False)

In [ ]:
halfdecade_counts = (
    dissolved_or_terminated_halfdecade_range
    .value_counts()
    .sort_index()
    .rename_axis("years")
    .reset_index(name="count")
)

halfdecade_counts.to_csv("docs/dissolved_terminated_by_halfdecade.csv", index=False)

In [ ]:
decade_counts.plot(x='years', y='count', kind="bar")

### When were the most stores incorporated?

In [ ]:
cleaned_incorporation_dates = df["DateOfIncorporation"].dropna()

# create decade groups
# // divides two numbers and rounds the result down to the nearest whole integer, so for ex. 1972 --> 1970. That means the "1970" bucket will represent 1970-1980
incorporation_decade_start = (cleaned_incorporation_dates.dt.year // 10) * 10
incorporation_decade_range = incorporation_decade_start.astype(str) + '-' + ((incorporation_decade_start + 9).astype(str))

incorporation_halfdecade = (cleaned_incorporation_dates.dt.year // 5) * 5
incorporation_halfdecade_range = incorporation_halfdecade.astype(str) + '-' + ((incorporation_halfdecade + 4).astype(str))

# graph decades and half decades

incorporation_decade_range.value_counts().sort_index().plot(kind="bar")
incorporation_halfdecade_range.value_counts().sort_index().plot(kind="bar")

Save incorporation info

In [ ]:

decade_counts = (
    incorporation_decade_range
    .value_counts()
    .sort_index()
    .rename_axis("years")
    .reset_index(name="count")
)

halfdecade_counts.to_csv('docs/incorporation_by_decade.csv', index=False)

halfdecade_counts = (
    incorporation_halfdecade_range
    .value_counts()
    .sort_index()
    .rename_axis("years")
    .reset_index(name="count")
)

decade_counts.to_csv('docs/incorporation_by_decade.csv', index=False)
halfdecade_counts.to_csv('docs/incorporation_by_halfdecade.csv', index=False)

How many are currently active?

In [ ]:
df.shape

In [ ]:
df["BusinessStatus"].value_counts()

### Poking around a bit more in the data

How many cities were at one point represented?

In [ ]:
## CAUTION -- THIS IS AN ESTIAMET BECAUSE THESE ARE OFFICE ADDRESSES AND SOME ARE OUTSIDE VT

# first title case them
df['PrincipalOfficeCity'] = df['PrincipalOfficeCity'].str.title()

# for reference -- non-VT

df_notVT = df[df['PrincipalOfficeState'] != 'VT']

df_notVT


#then count them 

df['PrincipalOfficeCity'].value_counts()

In [ ]:
df[df['BusinessName'] == 'PUTNEY GENERAL STORE']

### THIS IS IMPT: PUTNEY IS AN EX OF A STORE THAT IS INACTIVE - EXPIRED AND INACTIVE - CESSATED
# BUT NOT TERMINATED OR DISSOLVED. PUTNEY'S STORE IS STILL RUNNING

df[df['BusinessName'] == 'CALAIS GENERAL STORE']


### How many different general stores were active over time?

Make a new dataframe with just the info I want

First, I need to normalize some of the terms in the dataframe so that they are correctly recognized as the same store

In [ ]:
# Define which words should be treated as equivalent.
# Left side: variants you might see in the data. Right side: the standard form to use instead.

town_word_replacements = {
    'EAST ': '',
    'E ': '',
    'E.': '',
    'WEST ': '',
    'W ': '',
    'W.': '',
    'NORTH ': '',
    'N ': '',
    'N.': '',
    'SOUTH ': '',
    'S ': '',
    'S.': '',
    'ST ': 'Saint ',
    'ST. ': 'Saint ',
    'CENTER ': '',
    'CTR': '',
    'SO ': '',
    '.': '',

    # I need to sort nearby towns together to be able to group stores that are actually the same store
    # but listed in different towns next to eachother in different years

    'Marshfield': 'Marshfield/Plainfield',
    'Plainfield': 'Marshfield/Plainfield',
    'FILD': ''
    
}

business_word_replacements = {
    'THE': '',
    'LLC': '',
    'INC': '',
    'CORP': '',
    'LTD': '',
    'COUNTRY STORE': 'STORE',
    'GENERAL STORE': 'STORE',
    'HISTORICA': 'HISTORIC', # appears to be a typo in HISTORICA CRAFTSBURY GENERAL STORE
    "'S": "",
    ",": "",
    ".": "",
    "-": ""
}

# normalize business name

df["business_name_normalized"] = df["BusinessName"].str.upper()

for old, new in business_word_replacements.items():
    df["business_name_normalized"] = df["business_name_normalized"].str.replace(
        old, new, regex=False
    )

df["business_name_normalized"] = (
    df["business_name_normalized"]
      .str.replace(r"\s+", " ", regex=True)
      .str.strip()
)

# now for town name

df["town_name_normalized"] = df["PrincipalOfficeCity"].str.upper()

for old, new in town_word_replacements.items():
    df["town_name_normalized"] = df["town_name_normalized"].str.replace(
        old, new, regex=False
    )

df["town_name_normalized"] = (
    df["town_name_normalized"]
      .str.replace(r"\s+", " ", regex=True)
      .str.strip()
)


In [ ]:
df.head(50)

In [ ]:
# make a function to group stores by name,
# and then identify key dates (earliest initial filing date, most recent status, and record numbers) from each one

# pull out rows with no incorporation date
stores_with_no_incorporation_dates = df[df['DateOfIncorporation'].isna()]

# Now Keep only rows with valid dates for the groupby summary
df_valid_dates = df[df['DateOfIncorporation'].notna()]

# View your isolated records that need manual sorting
print(f"Isolated {len(stores_with_no_incorporation_dates)} records with missing dates.")

print(len(df_valid_dates))


In [ ]:
# Define function to pull out key info for new dataframe
def summarize_store_dates(group):

    # Safe idxmin call since NaNs are pre-filtered
    earliest_row = group.loc[group['DateOfIncorporation'].idxmin()]

    latest_row = group.loc[group['DateOfIncorporation'].idxmax()]

    # find the right date for last_date column
    # this will either be the dissolved or termination date if there is one, and then barring that,
    # the last annual report
    
    today = '08-03-2026'
    today = pd.to_datetime(today)
    
    if latest_row['BusinessStatus'] == 'Active - In Good Standing':
        type_of_last_date = "today"
        last_date = '07-31-2026' # today's date
        
    elif pd.notna(latest_row['DissolvedDate']):
        last_date = latest_row['DissolvedDate']
        type_of_last_date = "dissolved"
    elif pd.notna(latest_row['TerminationDate']):
        last_date = latest_row['TerminationDate']
        type_of_last_date = "termination"
    elif pd.notna(latest_row['LastAnnualReportDate']):
        last_date = latest_row['LastAnnualReportDate']
        type_of_last_date = "last_annual_report"
    elif pd.notna(latest_row['ExpirationDate']):
        if latest_row['ExpirationDate'] > today:
            last_date = today
            type_of_last_date = "today - expiration date after today"
        else:
            last_date = latest_row['ExpirationDate']
            type_of_last_date = "latest_expiration_date"
    else:
        last_date = None
        type_of_last_date = 'no_last_date'
    
    return pd.Series({
        'business_name_normalized': group.name[0],  # Extracts business_name from the group tuple
        'business_names': list(group['BusinessName']),
        'town_name_normalized': group.name[1],         # Extracts town from the group tuple
        'date_of_incorporation': earliest_row['DateOfIncorporation'],
        'most_recent_status': latest_row['BusinessStatus'],
        'record_numbers': list(group['RecordNumber']),
        'last_date': last_date,
        'type_of_last_date': type_of_last_date

    })

# Run the groupby on the valid data

df_new = df_valid_dates.groupby(['business_name_normalized', 'town_name_normalized']).apply(summarize_store_dates).reset_index(drop=True)

print(len(df_new))

In [ ]:
df_new.head(100)

# looking at how the last_date column got filled -- which type of date was used the most
df_new['type_of_last_date'].value_counts()

New column with range of active_dates

In [ ]:
# make sure in datetime
df_new["date_of_incorporation"] = pd.to_datetime(df_new["date_of_incorporation"])
df_new["last_date"] = pd.to_datetime(df_new["last_date"])

df_new[df_new['last_date'].isna()]

In [ ]:
df_new.shape

** CAUTION ** I am filtering out the four values without end dates, for now

In [ ]:
df_new = df_new[~df_new['last_date'].isna()]

df_new.shape

In [ ]:
df_new

In [ ]:
# now make the ranges
df_new["active_dates"] = df_new.apply(
    lambda row: pd.date_range(
        start=row["date_of_incorporation"],
        end=row["last_date"]
    ),
    axis=1
)

In [ ]:
years = range(
    df_new["date_of_incorporation"].dt.year.min(),
    df_new["last_date"].dt.year.max() + 1
)

active_by_year = pd.DataFrame({
    "year": years,
    "active_stores": [
        ((df_new["date_of_incorporation"].dt.year <= y) &
         (df_new["last_date"].dt.year >= y)).sum()
        for y in years
    ]
})

In [ ]:
active_by_year

Ok now group by decade

In [ ]:
type(active_by_year)

In [ ]:
active_by_year.plot(y='active_stores', x='year', kind='bar')

Given changes in filing practices over many decades, I'm zooming in to just after 1990, a decade or so before the first Dollar Store came to Vermont

In [ ]:
active_by_year_after1990 = active_by_year[active_by_year["year"] >= 1990]

active_by_year_after1990.plot(
    x="year",
    y="active_stores",
    kind="bar"
)


Export the info I need for my three charts to the docs folder

In [ ]:
active_by_year.to_csv('docs/stores_active_by_year.csv', index=False)

active_by_year_after1990.to_csv('docs/stores_active_by_year_after1990.csv', index=False)

Also need to export for map, just info for 2000 and 2026

In [ ]:
# for the year 2000

year_start_2000 = pd.Timestamp('2000-01-01')
year_end_2000 = pd.Timestamp('2000-12-31')

mask = (df_new['date_of_incorporation'] <= year_end_2000) & (df_new['last_date'] >= year_start_2000)
df_2000 = df_new[mask]

# and for the year 2025

year_start_2025 = pd.Timestamp('2025-01-01')
year_end_2025 = pd.Timestamp('2025-12-31')

mask = (df_new['date_of_incorporation'] <= year_end_2025) & (df_new['last_date'] >= year_start_2025)
df_2025 = df_new[mask]

df_2000['town_name_normalized'] = df_2000['town_name_normalized'] + ', VT'
df_2025['town_name_normalized'] = df_2025['town_name_normalized'] + ', VT'
# now filter for just the columns I need
df_filtered_2000 = df_2000[['business_names', 'town_name_normalized']]

df_filtered_2025 = df_2025[['business_names', 'town_name_normalized']]


In [ ]:
# geocode the business names

In [ ]:
import time
import geocoder
import os

from dotenv import load_dotenv

load_dotenv()
geocoder_api_key = os.getenv("geocoder_APIkey")

In [87]:
lats_2000 = []
longs_2000 = []

for index, row in df_filtered_2000.iterrows():
    g = geocoder.google(row['town_name_normalized'], key=geocoder_api_key)
    latlon = g.latlng
    lats_2000.append(latlon[0])
    longs_2000.append(latlon[1])
    time.sleep(0.1)

df_filtered_2000['lat'] = lats_2000
df_filtered_2000['lng'] = longs_2000

In [ ]:
lats_2025 = []
longs_2025 = []

for index, row in df_filtered_2025.iterrows():
    g = geocoder.google(row['town_name_normalized'], key=geocoder_api_key)
    latlon = g.latlng
    lats_2025.append(latlon[0])
    longs_2025.append(latlon[1])
    time.sleep(0.1)

df_filtered_2025['lat'] = lats_2025
df_filtered_2025['lng'] = longs_2025

ValueError: Length of values (285) does not match length of index (132)

In [83]:
# now save the csvs
df_filtered_2000.to_csv('docs/stores_active_in_2000.csv', index=False)
df_filtered_2025.to_csv('docs/stores_active_in_2025.csv', index=False)